In [ ]:
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import *

spark = (
            SparkSession
                .builder
                .master("local[*]")

                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://localhost:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark

In [ ]:
spark.sql("SHOW TABLES IN icebergtb.advworks_rdb ").show(truncate= False)
spark.sql(""" select * from icebergtb.advworks_rdb.raw_dimtransactioncode.manifests""").show(truncate=False)

In [ ]:
df = spark.sql(""" select row_number() over (order by businessentityid) as buyer_key, * from icebergtb.advworks_rdb.raw_dimbuyer""" ).show()

In [ ]:
buyer_temp = spark.sql(""" 
SELECT
    row_number() over (order by businessentityid) as buyer_key,
    businessentityid AS buyer_id,    firstname,
    lastname,                        birthdate,
    
    CAST(DATE_DIFF(CURRENT_DATE, birthdate) / 365 AS INT) AS age,

    CASE
        WHEN maritalstatus = 'M' THEN 'Married'
        ELSE 'Single'
    END AS maritalstatus,

    CASE
        WHEN gender = 'F' THEN 'Female'
        ELSE 'Male'
    END AS gender,

    hiredate,
    CAST(DATE_DIFF(CURRENT_DATE, hiredate) / 365 AS INT) AS years_in_company,
    departmentid,                    name AS department_name,
    groupname AS department_manager, jobtitle,
    payfrequency,                    rate,
    salarychange_date,               startdate,

    CASE
        WHEN enddate IS NULL AND startdate = salarychange_date THEN COALESCE(enddate, '2999-12-31')
        WHEN enddate IS NULL AND (DAY(startdate) - DAY(salarychange_date) = 1) THEN salarychange_date
        WHEN enddate IS NULL AND MAX(salarychange_date) < MAX(startdate) THEN COALESCE(enddate, '2999-12-31')
        ELSE enddate
    END AS enddate,

    salariedflag AS is_salaried,    currentflag AS is_current,
    extraced_date,
    
    COALESCE(modified_date, '2999-12-31') AS modified_date,
    
    eltversion
    
FROM icebergtb.advworks_rdb.raw_dimbuyer
GROUP BY
    businessentityid, firstname,            lastname,
    birthdate,        maritalstatus,        gender,
    hiredate,         departmentid,         name,
    groupname,        jobtitle,             payfrequency,
    rate,             salarychange_date,    startdate,
    enddate,          salariedflag,         currentflag,
    extraced_date,    modified_date,        eltversion
""").show()